<h1>Case Study 2</h1>

<h2>Importing Python Modules</h2>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import(classification_report, confusion_matrix, precision_recall_curve, roc_auc_score, average_precision_score, f1_score)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

<h2>Loading Dataset</h2>

In [2]:
df = pd.read_csv('creditcard.csv')
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


<h2>Train-Test Split (CRITICAL STEP BEFORE SMOTE)</h2>

In [3]:
X = df.drop(columns=['Class'])
y = df['Class']

# Stratify ensures both train and test maintain the rare fraud ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"\nTrain set fraud count: {y_train.sum()} / {len(y_train)}")
print(f"Test set fraud count:  {y_test.sum()} / {len(y_test)}")


Train set fraud count: 394 / 227845
Test set fraud count:  98 / 56962


<h2>Apply SMOTE (Synthetic Minority Over-sampling Technique)</h2>

In [4]:
# RULE: Apply SMOTE ONLY on X_train, NEVER on X_test (prevents data leakage)
smote = SMOTE(sampling_strategy=0.2, random_state=42)  
# sampling_strategy=0.2 brings minority class to 20% of majority class
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE Train shape: {X_train.shape}")
print(f"After SMOTE Train shape:  {X_train_resampled.shape}")
print(f"Resampled Train Fraud Count: {y_train_resampled.sum()}")

Before SMOTE Train shape: (227845, 30)
After SMOTE Train shape:  (272941, 30)
Resampled Train Fraud Count: 45490


<h2>Train XGBoost Classifier</h2>

In [5]:
xgb_model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.08,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train_resampled, y_train_resampled)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


<h2>Evaluate Default Threshold (0.50)</h2>

In [6]:
# Predict probabilities for Class 1 (Fraud)
y_probs = xgb_model.predict_proba(X_test)[:, 1]
default_preds = (y_probs >= 0.50).astype(int)

roc_auc = roc_auc_score(y_test, y_probs)
pr_auc = average_precision_score(y_test, y_probs)

print(f"\n--- Model Evaluation ---")
print(f"ROC-AUC Score: {roc_auc:.4f}")
print(f"PR-AUC (Average Precision): {pr_auc:.4f}")
print("\nConfusion Matrix at Default Threshold (0.50):")
print(confusion_matrix(y_test, default_preds))


--- Model Evaluation ---
ROC-AUC Score: 0.9793
PR-AUC (Average Precision): 0.8541

Confusion Matrix at Default Threshold (0.50):
[[56798    66]
 [   13    85]]


<h2>Decision Threshold Tuning</h2>

In [7]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

# Calculate F1-score for all thresholds to find the optimal trade-off
# (excluding last point where precision=1, recall=0)
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

print(f"\nOptimal Decision Threshold: {best_threshold:.4f}")
print(f"Best Test F1-Score: {best_f1:.4f}")

# Apply best threshold
tuned_preds = (y_probs >= best_threshold).astype(int)
print(f"\nConfusion Matrix at Tuned Threshold ({best_threshold:.3f}):")
print(confusion_matrix(y_test, tuned_preds))
print("\nClassification Report with Tuned Threshold:")
print(classification_report(y_test, tuned_preds, target_names=['Genuine', 'Fraud']))


Optimal Decision Threshold: 0.9856
Best Test F1-Score: 0.8398

Confusion Matrix at Tuned Threshold (0.986):
[[56857     7]
 [   22    76]]

Classification Report with Tuned Threshold:
              precision    recall  f1-score   support

     Genuine       1.00      1.00      1.00     56864
       Fraud       0.92      0.78      0.84        98

    accuracy                           1.00     56962
   macro avg       0.96      0.89      0.92     56962
weighted avg       1.00      1.00      1.00     56962



<h2>Feature Importance Interpretation</h2>

In [8]:
importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance_Gain': xgb_model.feature_importances_
}).sort_values(by='Importance_Gain', ascending=False)

print("\nXGBoost Feature Importance Scores:")
print(importance_df.to_string(index=False))


XGBoost Feature Importance Scores:
Feature  Importance_Gain
    V14         0.369357
    V10         0.284399
    V17         0.081893
     V4         0.036666
    V12         0.033447
     V3         0.020179
     V8         0.012269
    V28         0.011201
     V7         0.011089
    V13         0.010576
     V1         0.010382
    V11         0.009544
     V5         0.009180
    V20         0.008783
    V16         0.008398
     V9         0.007006
    V18         0.006640
    V15         0.006591
    V21         0.006301
     V2         0.006214
   Time         0.006001
 Amount         0.005804
    V22         0.005657
    V26         0.005337
    V25         0.005146
    V24         0.005096
    V19         0.004923
     V6         0.004346
    V27         0.003807
    V23         0.003767
